# PySpark Full Workflow Tutorial

This notebook walks through a complete PySpark data engineering workflow: creating DataFrames, reading/writing data, transformations, aggregations, joins, window functions, the Spark SQL API, and performance best practices. All code is compatible with Spark Connect (Serverless / Standard compute).

## 1. SparkSession — The Entry Point

The `SparkSession` is the unified entry point for all Spark functionality. In Databricks notebooks it is already created for you as the `spark` variable. Let's inspect it and check the Spark version.

In [0]:
# --- What is SparkSession? ---
# Think of SparkSession as your "control center" — it's the one object
# you use to talk to Spark. In Databricks, it's already created for you
# as the variable named "spark", so you don't need to set it up yourself.
print(f"Spark version: {spark.version}")

# SparkSession also lets you browse your data catalog (like a filing
# cabinet for all your tables) and check which database you're in.
print(f"Catalog: {spark.catalog.currentCatalog()}")
print(f"Database: {spark.catalog.currentDatabase()}")

# Let's list all the databases available in this workspace
spark.sql("SHOW DATABASES").show()

## 2. Creating DataFrames from In-Memory Data

You can create DataFrames directly from Python lists, dictionaries, or pandas DataFrames using `spark.createDataFrame()`. This is great for prototyping and testing.

In [0]:
from pyspark.sql import Row
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType, DateType
)

# --- Method 1: From a list of Row objects ---
# A Row is like a single record in a spreadsheet — each one holds
# a set of named values (like id, name, salary). We give Spark a
# list of these rows, and it builds a DataFrame (a table) from them.
rows = [
    Row(id=1, name="Alice",   department="Engineering", salary=95000,  hire_date="2021-03-15"),
    Row(id=2, name="Bob",     department="Marketing",    salary=72000,  hire_date="2020-07-01"),
    Row(id=3, name="Charlie", department="Engineering", salary=110000, hire_date="2019-01-20"),
    Row(id=4, name="Diana",   department="HR",           salary=65000,  hire_date="2022-11-05"),
    Row(id=5, name="Eve",     department="Marketing",    salary=78000,  hire_date="2021-09-12"),
    Row(id=6, name="Frank",   department="Engineering", salary=88000,  hire_date="2020-04-18"),
    Row(id=7, name="Grace",   department="HR",           salary=70000,  hire_date="2023-02-28"),
    Row(id=8, name="Heidi",   department="Marketing",    salary=82000,  hire_date="2019-06-30"),
]
employees = spark.createDataFrame(rows)

# --- Method 2: From a list of tuples with an explicit schema ---
# Sometimes you want full control over the column names and data types.
# You provide a "schema" (a blueprint) that tells Spark exactly what
# each column is called and what type of data it holds.
# This is like defining the column headers and data types in a
# spreadsheet before you start entering data.
schema = StructType([
    StructField("emp_id",      IntegerType(), False),
    StructField("project",     StringType(),  True),
    StructField("emp_ref",     IntegerType(), True),   # foreign key to employees.id
    StructField("hours_week",   IntegerType(), True),
    StructField("start_date",  StringType(),  True),
])

project_data = [
    (101, "Phoenix",   1, 40, "2024-01-15"),
    (102, "Phoenix",   3, 35, "2024-01-15"),
    (103, "Quantum",   2, 20, "2024-02-01"),
    (104, "Quantum",   5, 30, "2024-02-01"),
    (105, "Phoenix",   6, 25, "2024-03-10"),
    (106, "Stellar",   4, 15, "2024-04-01"),
    (107, "Stellar",   7, 10, "2024-04-01"),
    (108, "Quantum",   8, 40, "2024-03-15"),
]
projects = spark.createDataFrame(project_data, schema=schema)

# .count() is an "action" — it tells Spark to actually compute a result
# (up to this point, everything was just instructions waiting to run).
print(f"employees row count: {employees.count()}")
print(f"projects row count:  {projects.count()}")
employees.show()

## 3. Inspecting DataFrames — Schema, Summary, and Sampling

Before transforming data, always understand its schema and distribution. Spark Connect uses lazy schema analysis, so calling `printSchema()` or `columns` early helps catch errors before they surface at action time.

In [0]:
# --- What's a schema? ---
# A schema describes the structure of your data: column names, data types,
# and whether a column can be empty (nullable). It's like the column
# headers in a spreadsheet, but with explicit data types (string, integer, date).
print("=== employees schema ===")
employees.printSchema()

print("=== projects schema ===")
projects.printSchema()

# Summary statistics give you a quick overview of numeric columns:
# count, average, standard deviation, min, and max values.
print("=== salary summary ===")
employees.describe("salary").show()

# You can list column names as a simple Python list
print(f"employees columns: {employees.columns}")

# Sampling lets you peek at a random subset of rows without loading
# the entire dataset — useful for getting a feel for the data quickly.
# The 'seed' makes the random selection reproducible (same result each run).
print("=== sample rows ===")
employees.sample(fraction=0.5, seed=42).show()

## 4. Basic Transformations — Select, Filter, WithColumn, Drop

Transformations are **lazy** — they build a logical plan but don't execute until an action (like `show()`, `collect()`, or `write()`) is triggered. This allows Spark to optimize the entire pipeline.

In [0]:
from pyspark.sql.functions import col, when, lit

# --- SELECT: choose specific columns ---
# Like picking certain columns in a spreadsheet, select() keeps only
# the columns you name and drops the rest.
employees.select("id", "name", "department").show(3)

# --- FILTER: keep only the rows you care about ---
# filter() is like applying a filter in a spreadsheet — it keeps only
# the rows where the condition is True, and discards the rest.
eng_only = employees.filter(col("department") == "Engineering")
eng_only.show()

# You can combine multiple conditions with & (AND) and | (OR),
# just like in a spreadsheet's advanced filter.
high_paid = employees.filter((col("salary") >= 80000) & (col("department") == "Engineering"))
high_paid.show()

# --- WITHCOLUMN: add a new column or overwrite an existing one ---
# withColumn() is like adding a new column to a spreadsheet based on a
# formula. You give it a column name and a formula (expression), and
# Spark computes the values for every row.
from pyspark.sql.functions import to_date, datediff, current_date

employees_typed = employees.withColumn("hire_date", to_date(col("hire_date"), "yyyy-MM-dd"))

# Calculate how many days each employee has been with the company.
# datediff(a, b) returns the number of days between a and b.
employees_typed = employees_typed.withColumn(
    "tenure_days", datediff(current_date(), col("hire_date"))
)

# Categorize salaries into tiers using when/otherwise.
# This works like a series of IF/ELSE statements:
#   IF salary >= 100,000 THEN "Senior"
#   ELSE IF salary >= 75,000 THEN "Mid"
#   ELSE "Junior"
employees_typed = employees_typed.withColumn(
    "salary_tier",
    when(col("salary") >= 100000, "Senior")
    .when(col("salary") >= 75000, "Mid")
    .otherwise("Junior")
)

# --- DROP: remove a column you no longer need ---
# This simply removes the column from the DataFrame to keep things tidy.
employees_typed = employees_typed.drop("hire_date")

display(employees_typed)

## 5. String and Date Functions

PySpark provides hundreds of built-in column functions. Here are some of the most common ones for string manipulation and date arithmetic.

In [0]:
from pyspark.sql.functions import (
    upper, lower, concat, concat_ws, substring, length, trim, regexp_replace,
    date_format, year, month, dayofweek, add_months, months_between, last_day
)

# Create a working copy. The .alias("e") gives the DataFrame a
# short nickname so we can refer to its columns clearly later (e.g. e.name).
df = employees_typed.alias("e")

# --- String functions (text manipulation) ---
# These work just like spreadsheet text functions (UPPER, TRIM, etc.):
#   upper()       — convert text to ALL CAPS
#   trim()        — remove extra spaces at the start/end
#   length()      — count how many characters are in the text
#   concat_ws()   — glue multiple columns together with a separator
#                   (like "Alice - Engineering")
df = df.withColumn("name_upper", upper(col("name"))) \
       .withColumn("dept_clean", trim(col("department"))) \
       .withColumn("name_length", length(col("name"))) \
       .withColumn("label", concat_ws(" - ", col("name"), col("department")))

# --- Date functions (calendar operations) ---
# These are like spreadsheet date functions (YEAR(), MONTH(), etc.).
# We start from the original 'employees' DataFrame because we dropped
# hire_date earlier — here we need it again.
emp_dates = employees.withColumn("hire_date", to_date(col("hire_date"), "yyyy-MM-dd"))

# Extract useful pieces from the date:
#   year()           — the year (e.g. 2021)
#   month()          — the month number (1-12)
#   dayofweek()      — the day of week (1=Sunday, 7=Saturday)
#   date_format()    — display the month name (e.g. "Mar")
#   months_between() — how many months have passed since that date
#   add_months()     — shift a date forward by N months
#   last_day()       — the last day of that month (useful for billing cycles)
emp_dates = emp_dates.withColumn("hire_year",  year(col("hire_date"))) \
                     .withColumn("hire_month", month(col("hire_date"))) \
                     .withColumn("hire_dow",   dayofweek(col("hire_date"))) \
                     .withColumn("hire_month_name", date_format(col("hire_date"), "MMM")) \
                     .withColumn("months_employed", months_between(current_date(), col("hire_date")).cast("int")) \
                     .withColumn("review_date", last_day(add_months(col("hire_date"), 3)))

display(emp_dates.select("id", "name", "hire_date", "hire_year", "hire_month_name", "months_employed", "review_date"))

## 6. Aggregations — GroupBy, Pivot, and Rollup

Aggregations are the backbone of analytics. `groupBy()` followed by `agg()` lets you compute multiple metrics at once.

In [0]:
from pyspark.sql.functions import count, sum as spark_sum, avg, min as spark_min, max as spark_max, round as spark_round, countDistinct

# --- Group by department, then compute summary numbers ---
# groupBy() splits the rows into groups (one per department), like a
# pivot table in a spreadsheet. Then agg() calculates one or more
# summary values for each group — count, average, min, max, etc.
dept_summary = employees_typed.groupBy("department").agg(
    count("*").alias("headcount"),
    spark_round(avg("salary"), 2).alias("avg_salary"),
    spark_min("salary").alias("min_salary"),
    spark_max("salary").alias("max_salary"),
    countDistinct("salary_tier").alias("tier_diversity"),
)
dept_summary.orderBy(col("avg_salary").desc()).show()

# --- Pivot: turn a row value into columns ---
# A pivot is like a cross-tab in a spreadsheet. Instead of listing
# "Junior", "Mid", "Senior" as rows, it turns each salary tier into
# its own column, with the average salary as the cell value.
pivot_df = employees_typed.groupBy("department").pivot("salary_tier").agg(
    spark_round(avg("salary"), 0).alias("avg_salary")
)
pivot_df.show()

# --- Rollup: subtotals at multiple levels (like Excel subtotals) ---
# rollup() produces not just per-group totals, but also grand totals.
# You'll see rows with NULL for department or salary_tier — those
# are the sub-total or grand-total rows.
rollup_df = employees_typed.rollup("department", "salary_tier").agg(
    count("*").alias("headcount"),
    spark_sum("salary").alias("total_salary"),
)
rollup_df.orderBy("department", "salary_tier", ascending=[True, True]).show()

## 7. Joins — Combining DataFrames

Joins merge two DataFrames on a key. Spark supports inner, left, right, outer, left_semi, and left_anti joins. For small tables, use `broadcast()` to force a broadcast join and avoid shuffles.

In [0]:
from pyspark.sql.functions import broadcast

# A join combines two tables side by side, matching rows that share
# a common value — just like VLOOKUP in a spreadsheet. Here we match
# each employee (by their id) to their project assignments (by emp_ref).
joined = employees_typed.alias("e").join(
    projects.alias("p"),
    col("e.id") == col("p.emp_ref"),
    "inner"
)

# After joining, we pick the most useful columns from both tables
# so the result is easy to read.
joined.select(
    col("e.id"), col("e.name"), col("e.department"),
    col("p.project"), col("p.hours_week"), col("e.salary_tier")
).orderBy("e.id").show()

# --- Left join: keep EVERY employee, even those without a project ---
# In an "inner" join, employees without a project would disappear.
# A "left" join keeps all rows from the left table (employees) and
# fills in NULL for the right table's columns when there's no match.
# Think of it as "keep all employees, and add project info if available."
left_joined = employees_typed.alias("e").join(
    projects.alias("p"),
    col("e.id") == col("p.emp_ref"),
    "left"
).withColumn("has_project", col("p.project").isNotNull())

left_joined.groupBy("has_project").agg(count("*").alias("count")).show()

# --- Broadcast join: a performance shortcut for small tables ---
# Normally a join requires shuffling data between workers (slow).
# If one table is small enough, broadcast() sends a complete copy to
# every worker so they can do the join locally — no shuffling needed.
# Think of it as handing everyone a printed copy of a small lookup table
# instead of making them walk to a filing cabinet to look things up.
broadcast_joined = employees_typed.alias("e").join(
    broadcast(projects).alias("p"),
    col("e.id") == col("p.emp_ref"),
    "inner"
)
print(f"Broadcast join result count: {broadcast_joined.count()}")

## 8. Window Functions — Analytics Across Rows

Window functions operate over a "frame" of rows related to the current row. They are essential for running totals, rankings, and lag/lead comparisons.

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, rank, dense_rank, lag, lead, sum as spark_sum, avg, percent_rank

# --- What is a "window"? ---
# A window defines which rows to compare the current row against.
# Think of it like grouping, but instead of collapsing each group into
# one row, the window keeps every row and lets you compute values
# relative to the other rows in the same group (like a running total
# or a rank within a department).
#
# partitionBy = split into groups (like groupBy)
# orderBy     = sort rows within each group
# rowsBetween = which rows to include in the calculation (a "frame")
window_dept_by_salary = Window.partitionBy("department").orderBy(col("salary").desc())
window_dept_running = Window.partitionBy("department").orderBy("salary").rowsBetween(Window.unboundedPreceding, Window.currentRow)
window_dept_all = Window.partitionBy("department")

# Now we apply several window functions at once. Each one computes a
# value by looking at the rows in the same department (the window).
# This gives us ranking, running totals, and comparisons to the
# previous/next employee — all in a single pass.
windowed = employees_typed.select(
    "id", "name", "department", "salary", "salary_tier",
    # Ranking within department by salary
    row_number().over(window_dept_by_salary).alias("rank_in_dept"),
    dense_rank().over(window_dept_by_salary).alias("dense_rank_in_dept"),
    percent_rank().over(window_dept_by_salary).alias("pct_rank"),
    # Running total of salary within department (ascending)
    spark_sum("salary").over(window_dept_running).alias("running_salary_total"),
    # Department average salary alongside each row
    spark_round(avg("salary").over(window_dept_all), 0).alias("dept_avg_salary"),
    # Previous and next salary in the department ranking
    lag("salary", 1).over(window_dept_by_salary).alias("prev_salary"),
    lead("salary", 1).over(window_dept_by_salary).alias("next_salary"),
)

windowed.orderBy("department", "rank_in_dept").show(20)

## 9. Spark SQL API — Querying DataFrames with SQL

The `SparkSession.sql()` method lets you run standard SQL against any registered view or catalog table. This is useful for analysts who prefer SQL or for reusing existing SQL logic.

In [0]:
# --- What is a "temp view"? ---
# A temporary view lets you give a DataFrame a name that you can use
# inside SQL queries. It's like creating a bookmark to a DataFrame so
# you can refer to it with SQL instead of Python. It only exists for
# this notebook session — it's not saved anywhere.
employees_typed.createOrReplaceTempView("v_employees")
projects.createOrReplaceTempView("v_projects")

# --- Simple SQL query ---
# Now that we have temp views, we can write standard SQL just like
# we would against any database table.
spark.sql("""
    SELECT department, COUNT(*) AS headcount, ROUND(AVG(salary), 0) AS avg_salary
    FROM v_employees
    GROUP BY department
    ORDER BY avg_salary DESC
""").show()

# --- SQL with a join ---
# The same join we did in Python (cell 7), but written as SQL instead.
# Some people find SQL easier to read for complex queries.
spark.sql("""
    SELECT 
        e.name, 
        e.department, 
        e.salary_tier,
        p.project,
        p.hours_week
    FROM v_employees e
    INNER JOIN v_projects p ON e.id = p.emp_ref
    ORDER BY e.name
""").show()

# --- SQL with a window function ---
# Window functions also work in SQL using OVER(PARTITION BY ...).
# This is the SQL version of what we did with the Window class in cell 8.
spark.sql("""
    SELECT 
        name,
        department,
        salary,
        RANK() OVER (PARTITION BY department ORDER BY salary DESC) AS dept_rank
    FROM v_employees
    ORDER BY department, dept_rank
""").show()

## 10. Writing Data — Delta Tables

Delta Lake is the default storage format in Databricks. It provides ACID transactions, schema enforcement, and time travel. Use `write.saveAsTable()` to persist to the Unity Catalog.

In [0]:
# --- What is a Delta table? ---
# A Delta table is a permanent table stored in Databricks. Unlike a
# DataFrame (which only lives in memory while your code runs), a Delta
# table is saved to disk and can be queried later by anyone in your
# workspace — even from a different notebook.
#
# First, let's figure out where to save it (the current catalog/schema).
write_catalog = spark.catalog.currentCatalog()
write_schema = f"{write_catalog}.{spark.catalog.currentDatabase()}"
print(f"Writing to: {write_schema}")

# Write our cleaned-up employees data to a Delta table.
# mode("overwrite") means: if the table already exists, replace it entirely.
# This is the simplest write mode for tutorials — in production you'd
# typically use "append" or "merge" to avoid losing existing data.
employees_typed.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{write_schema}.tutorial_employees")

# Do the same for the joined employees + projects table
joined.write \
    .mode("overwrite") \
    .saveAsTable(f"{write_schema}.tutorial_emp_projects")

# Read it back to make sure it was saved correctly.
# Because it's a permanent table now, we can query it with SQL.
spark.sql(f"SELECT * FROM {write_schema}.tutorial_employees LIMIT 5").show()

# --- Delta Time Travel ---
# One special feature of Delta: it remembers every change made to the
# table. You can look back at the full history of operations — who wrote
# what, when. This is like a version history for your data.
print(f"Delta table history:")
spark.sql(f"DESCRIBE HISTORY {write_schema}.tutorial_employees").select("version", "timestamp", "operation").show(3)

## 11. Performance Best Practices

Spark performance comes from understanding **lazy evaluation**, the difference between **narrow** and **wide** transformations, and avoiding common pitfalls.

**What are narrow and wide transformations?**

| Type | Examples | What Happens |
| --- | --- | --- |
| **Narrow** | `select`, `filter`, `withColumn` | Each row stays on its own worker — no data moves between workers. Fast and cheap. |
| **Wide** | `repartition`, `groupBy`, `join`, `distinct` | Data must be **shuffled** across the network so matching rows end up on the same worker. This is expensive and is the #1 cause of slow Spark jobs. |

Think of it this way: a *narrow* transformation is like each person in a team highlighting rows on their own stack of papers — no coordination needed. A *wide* transformation is like everyone needing to sort all papers by color — they have to pass papers to each other over the network first.

Here are key tips with demonstrations.

In [0]:
from pyspark.sql.functions import spark_partition_id, rand

# --- Tip 1: Don't pull all data to your screen at once ---
# collect() grabs EVERY row from the cluster and loads it into the
# driver (the single machine running your notebook). If your table has
# millions of rows, this can crash your notebook!
# Instead, use take(n) to grab just a few rows for inspection.
sample_rows = employees_typed.take(3)
print(f"Collected {len(sample_rows)} rows to driver (safe with take)")

# --- Tip 2: Caching — keep frequently-used data in memory ---
# If you use the same DataFrame several times, caching it avoids
# re-computing it from scratch each time (like keeping a spreadsheet
# open instead of re-opening the file each time you need it).
#
# NOTE: On Serverless compute, .cache() / .persist() is NOT available
# because the worker machines are temporary (they appear and disappear
# as needed, so there's no permanent memory to cache into).
# On Standard / Assigned compute, you would use:
#   cached_df = employees_typed.cache()          # saves a copy in memory
#   cached_df.count()                            # forces Spark to actually build it
#   cached_df.groupBy("salary_tier").count().show()  # reads from the saved copy
#   cached_df.unpersist()                        # releases the memory when done
# On Serverless, Spark's Adaptive Query Engine (AQE) and Delta's built-in
# caching handle this for you automatically.
print("Caching (.cache/.persist) is not available on Serverless compute.")
print("On Standard compute, use .cache() for DataFrames reused multiple times.")

# --- Tip 3: Broadcast small tables to avoid expensive data shuffling ---
# Already demonstrated in the Joins section (cell 7).
# When you join two tables, Spark normally has to shuffle (physically
# move) data between worker machines to match up the rows. This is
# slow, especially on large datasets. If one table is small, broadcast()
# sends a full copy to every worker so they can do the join locally
# — no shuffling needed.

# --- Tip 4: Check how your data is split across workers ---
# Spark splits large datasets into "partitions" — chunks of rows that
# different workers process in parallel. If one partition is much larger
# than the others (a "skew"), that worker becomes a bottleneck while
# the others sit idle. spark_partition_id() tells you which partition
# each row landed in, so you can check the distribution.
part_info = employees_typed.withColumn("partition_id", spark_partition_id())
part_info.groupBy("partition_id").count().show()

# --- Tip 5: Repartition (a "wide" transformation) vs. Coalesce ---
#
# What is a "narrow" vs. "wide" transformation?
#
# NARROW transformation (e.g. select, filter, withColumn):
#   Each row stays on the same worker — no data needs to move between
#   workers. Think of it like each person in a team working on their own
#   stack of papers without talking to anyone else.
#
# WIDE transformation (e.g. repartition, groupBy, join, distinct):
#   Data must be physically reshuffled across the network so that all
#   rows with the same key end up on the same worker. Think of it like
#   sorting a deck of cards that was dealt to 4 people — everyone has to
#   pass cards around until each person holds only one suit. This
#   network shuffling is expensive and is the most common cause of slow
#   Spark jobs.
#
# repartition(n, column) is a WIDE transformation:
#   It reshuffles ALL rows across the network so that rows with the same
#   value of the given column land in the same partition. Here we ask
#   Spark to create 4 partitions, grouping by department.
repartitioned = employees_typed.repartition(4, "department")

# Because repartition is a wide transformation, it triggers a shuffle.
# We can see the result by checking how many rows ended up in each
# partition using spark_partition_id().
part_dist = repartitioned.withColumn("pid", spark_partition_id()).groupBy("pid").count()
print("After repartition (4 partitions by department) — row distribution:")
part_dist.show()

# coalesce(n) is a NARROW-ish operation — it MERGES existing partitions
# into fewer ones without a full shuffle. It's much cheaper than
# repartition because it just combines adjacent partitions.
# Use it when you want to reduce the number of partitions (e.g. before
# writing a small output file) without paying the cost of a full shuffle.
coalesced = repartitioned.coalesce(1)
coalesced_dist = coalesced.withColumn("pid", spark_partition_id()).groupBy("pid").count()
print("After coalesce(1) — all rows merged into 1 partition:")
coalesced_dist.show()

# --- Tip 6: Use explain() to see what Spark actually does under the hood ---
# explain() prints the "physical plan" — the step-by-step recipe Spark
# will follow to execute your query. It shows things like whether a join
# will use broadcasting or shuffling, and whether the Photon engine
# (Databricks' fast execution engine) is being used. Reading the plan
# helps you spot bottlenecks (e.g. an unexpected shuffle).
print("=== Physical Plan ===")
joined.explain()

## 12. Cleaning Up Tutorial Tables

Run this cell to drop the tutorial tables created during the workflow.

In [0]:
# --- Cleaning up ---
# The cells above created two permanent Delta tables in your workspace.
# When you're done with this tutorial, uncomment the lines below (remove
# the # at the start of each line) and run this cell to delete them.
# This keeps your workspace tidy and frees up storage.
#
# write_schema = f"{spark.catalog.currentCatalog()}.{spark.catalog.currentDatabase()}"
# spark.sql(f"DROP TABLE IF EXISTS {write_schema}.tutorial_employees")
# spark.sql(f"DROP TABLE IF EXISTS {write_schema}.tutorial_emp_projects")
# print("Tutorial tables dropped.")
print("Cleanup cell ready — uncomment the DROP statements to remove tutorial tables.")

---
## Summary

This tutorial covered the full PySpark workflow:

1. **SparkSession** — the entry point (pre-created in Databricks)
2. **Creating DataFrames** — from in-memory data with explicit schemas
3. **Inspecting Data** — `printSchema()`, `describe()`, `sample()`
4. **Transformations** — `select`, `filter`, `withColumn`, `when/otherwise`, `drop`
5. **String & Date Functions** — `upper`, `concat_ws`, `date_format`, `months_between`, etc.
6. **Aggregations** — `groupBy().agg()`, `pivot()`, `rollup()`
7. **Joins** — inner, left, and broadcast joins
8. **Window Functions** — `row_number`, `rank`, `lag/lead`, running totals
9. **SQL API** — `spark.sql()` against registered temp views
10. **Writing Delta** — `saveAsTable()`, schema enforcement, time travel
11. **Performance** — caching, broadcast, repartition/coalesce, `explain()`

### Key Takeaways
- Transformations are **lazy**; actions trigger execution
- Prefer the **DataFrame API** over RDDs (especially on Spark Connect / Serverless)
- Use `broadcast()` for small-table joins to avoid shuffles
- Cache only DataFrames that are **reused** multiple times
- Always `printSchema()` early to catch column reference errors (Spark Connect lazy schema analysis)